In [ ]:
# Practice Problems Day 9

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 5 assumes you have done them. Plan on about 45 minutes.

**Reading:** Geron chapter 9, **From Biological to Artificial Neurons**, and chapter 10, **PyTorch Fundamentals**.

GOALS:

1. Write gradient descent yourself, first for one number and then for a whole neuron.
2. Train a neuron on real data and check it against scikit-learn.
3. Explain in words what a trained neuron learned.

**How much of this should you write yourself?** Problem 1 gives you most of the code, with comments telling you what to write on the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 9 practice"
git push
```

---

## Problem 1: Gradient descent on one number

**What to copy, what to write:** Most of the code is given. Write the lines the comments ask for.

Before we train a whole neuron, let's do gradient descent on the simplest problem there is. We want the value of $w$ that makes

$$g(w) = (w - 4)^2$$

as small as possible. You already know the answer is $w = 4$, so we can check that gradient descent finds it.

**1a.** By hand, find $g'(w)$ and then $g'(1)$. Then run the cell below and check that autograd agrees with you.

In [ ]:
import torch

w = torch.tensor(1.0, requires_grad=True)

# Write g(w) = (w - 4)^2, then call .backward() on it


print("g'(1) from autograd:", w.grad)

**1b.** Now gradient descent. Start at $w = 0$ and take 50 steps with a learning rate of 0.1. Where does $w$ end up?

**1c.** Change the learning rate to **1.1** and run it again. What does $w$ do? (Hint: work out by hand where the first step from $w = 0$ lands.)

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

for step in range(50):
    # compute g = (w - 4)^2
    g = (w - 4) ** 2

    # call backward to find the derivative


    # take a step downhill (inside torch.no_grad())


    # clear the gradient


    if step % 10 == 0:                 # % is the remainder, so this is True for steps 0, 10, 20, ...
        print("step", step, "  w =", round(w.item(), 4))

print("final w:", round(w.item(), 4))

*Your answers here.*

---

## Problem 2: Train a neuron on the penguins (steps outlined)

**What to copy, what to write:** The first cell is written for you, just run it. In the second cell, write the loop yourself. It is the same pattern as the notes, so type it out rather than pasting it.

The data is the penguins from the Day 8 homework. We use the four body measurements to predict a penguin's **sex** (1 for male, 0 for female).

**2a.** Train a neuron with gradient descent: start every weight and the bias at 0, use a learning rate of **0.1**, and take **2000 steps**. Plot the loss at every step, with scikit-learn's loss as a dashed line.

**2b.** Did your neuron reach scikit-learn's loss after 2000 steps? Compare the validation accuracy of your neuron and scikit-learn.

In [ ]:
# RUN THIS. Load, split, scale, tensors, and scikit-learn's model, all like the notes.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

penguins = pd.read_csv("data/penguins.csv").dropna()
measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = penguins[measurements]
y = (penguins["sex"] == "Male").astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)

# scikit-learn's model in a full pipeline, fit on the raw rows
log_reg_pipeline = Pipeline([("prep", preprocessing), ("model", LogisticRegression(C=np.inf))])   # C=np.inf: no regularization
log_reg_pipeline.fit(X_train, y_train)
log_reg = log_reg_pipeline.named_steps["model"]          # the fitted model inside the pipeline
p_sklearn = torch.tensor(log_reg.predict_proba(X_train_scaled)[:, 1], dtype=torch.float32)
loss_sklearn = F.binary_cross_entropy(p_sklearn, y_train_t)

print("training penguins:", X_train_t.shape[0], "   validation penguins:", X_valid_t.shape[0])
print("scikit-learn's loss:", round(loss_sklearn.item(), 4))

In [ ]:
# Step 1 (2a): w (4 zeros) and b (1 zero), both with requires_grad=True. learning_rate = 0.1, and an empty losses list


# Step 2 (2a): the loop, 2000 steps: predict, loss (save it), backward, update inside torch.no_grad(), clear


# Step 3 (2a): plot losses, with scikit-learn's loss as a dashed line: plt.axhline(loss_sklearn.item(), linestyle="--")


# Step 4 (2b): predict the validation penguins with your neuron (threshold 0.5), then print both validation accuracies
#              scikit-learn's predictions are log_reg_pipeline.predict(X_valid)


*Your answers here.*

---

## Problem 3: What did the neuron learn? (words)

Run the cell below. It prints your neuron's four weights next to the name of each measurement.

**3a.** Which measurement pushes the neuron most strongly toward "male"?

**3b.** Explain to a friend who took DATA 101 but not this class what your neuron does with a new penguin, from its four measurements to the word "male" or "female". Use the words weight, sum, sigmoid and threshold, and no code. Three or four sentences.

In [ ]:
# the weights from Problem 2, one per measurement
print(pd.Series(w.detach().numpy(), index=measurements).round(3))
print("bias:", round(b.item(), 3))

*Your answers here.*

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** Train the penguin neuron with the learning rates 0.01, 0.1, 1.0 and 20, and plot all four loss curves on one plot. Which one would you use? What goes wrong with 20?

**B.** Start the weights at random numbers, `torch.randn(4, requires_grad=True)`, instead of zeros. Does the neuron end up at the same loss?

In [ ]:
# your code here

*Your answers here.*